# Идентификация автора по стилометрическим признакам

**Задача:** определить автора текста по символьным n-граммам и проверить устойчивость модели к опечаткам.

**Стек:** Hugging Face Datasets, scikit-learn, SciPy, pandas.

TF-IDF обучается только на тренировочных текстах. Символьные n-граммы фиксируют пунктуацию, окончания слов и устойчивые стилевые шаблоны.


In [ ]:
import random
import time

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

SEED = 42
MAX_ROWS = 20_000
TOP_AUTHORS = 50
random.seed(SEED)


In [ ]:
dataset = load_dataset("Hieuman/blog_authorship-doc", split=f"train[:{MAX_ROWS}]")
data = dataset.to_pandas()[["fullText", "authorIDs"]].dropna()
data = data.drop_duplicates("fullText")

selected_authors = data["authorIDs"].value_counts().head(TOP_AUTHORS).index
data = data[data["authorIDs"].isin(selected_authors)].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    data["fullText"],
    data["authorIDs"],
    test_size=0.20,
    random_state=SEED,
    stratify=data["authorIDs"],
)
print(f"Текстов: {len(data):,}; авторов: {data['authorIDs'].nunique()}")


In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

classifier = LinearSVC(C=1.0, class_weight="balanced", random_state=SEED)
classifier.fit(X_train_vectorized, y_train)


In [ ]:
started = time.perf_counter()
prediction = classifier.predict(X_test_vectorized)
elapsed = time.perf_counter() - started

metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, prediction),
        "macro_f1": f1_score(y_test, prediction, average="macro"),
        "milliseconds_per_text": elapsed / len(y_test) * 1_000,
    }
)
metrics.round(4).to_frame("value")


In [ ]:
def add_typo_noise(text: str, probability: float = 0.03) -> str:
    letters = list(str(text))
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    for index, character in enumerate(letters):
        if character.isalpha() and random.random() < probability:
            letters[index] = random.choice(alphabet)
    return "".join(letters)


random.seed(SEED)
noisy_test = X_test.map(add_typo_noise)
noisy_prediction = classifier.predict(vectorizer.transform(noisy_test))

robustness = pd.Series(
    {
        "clean_accuracy": accuracy_score(y_test, prediction),
        "noisy_accuracy": accuracy_score(y_test, noisy_prediction),
        "accuracy_drop": accuracy_score(y_test, prediction)
        - accuracy_score(y_test, noisy_prediction),
    }
)
robustness.round(4).to_frame("value")


In [ ]:
sample_size = min(1_000, X_test_vectorized.shape[0])
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(X_test_vectorized.shape[0], size=sample_size, replace=False)
sample_vectors = X_test_vectorized[sample_indices]
sample_labels = y_test.iloc[sample_indices].to_numpy()

stylometric_separation = silhouette_score(
    sample_vectors,
    sample_labels,
    metric="cosine",
)
print(f"Silhouette score по авторам: {stylometric_separation:.4f}")


In [ ]:
feature_names = vectorizer.get_feature_names_out()
rows = []
for class_index, author in enumerate(classifier.classes_[:5]):
    top_indices = np.argsort(classifier.coef_[class_index])[-10:][::-1]
    rows.append(
        {
            "author": author,
            "character_patterns": feature_names[top_indices].tolist(),
        }
    )
pd.DataFrame(rows)


## Ограничения

Модель может запоминать тему или платформенные шаблоны, а не только авторский стиль. Для честной проверки нужны временное разбиение и перенос на другой домен. Результаты нельзя использовать как единственное доказательство авторства.
